# 03 · Inference

Load the packaged model exactly as the API does and score new patients.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
%matplotlib inline

In [2]:
from cardiorisk.predict import RiskModel
model = RiskModel()
print(model.name, model.version)

logistic_regression 1.0.0+202610030813


In [3]:
patients = [
  dict(age=58, sex=1, cp=4, trestbps=140, chol=260, fbs=0, restecg=2, thalach=120, exang=1, oldpeak=2.4, slope=2, ca=2, thal=7),
  dict(age=41, sex=0, cp=2, trestbps=126, chol=306, fbs=0, restecg=0, thalach=163, exang=0, oldpeak=0.0, slope=1, ca=0, thal=3),
  dict(age=63, sex=1, cp=3, trestbps=145, chol=233, fbs=1, restecg=2, thalach=150, exang=0, oldpeak=2.3, slope=3, ca=None, thal=None),
]
import pandas as pd
pd.DataFrame([p.__dict__ for p in model.predict(patients)])

,prediction,label,probability_disease,confidence,risk_band
0,1,disease,0.9969,0.9969,high
1,0,no_disease,0.0353,0.9647,low
2,0,no_disease,0.1641,0.8359,low


The third patient has unknown `ca`/`thal`: the pipeline's imputers fill them with the training mode, so the model still returns a prediction. The same request shape is accepted by `POST /predict`.

In [4]:
import mlflow.pyfunc
from cardiorisk import config
pyfunc = mlflow.pyfunc.load_model(str(config.MODELS_DIR / 'mlflow_model'))
pyfunc.predict(pd.DataFrame(patients[:2]).astype(float))

2026/10/03 13:43:55 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/flowshiftt/Desktop/gmlops/.venv/lib/python3.14/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


array([1, 0])